# T-SOHO on Colab — CIFAR-100
This notebook runs the repository implementation. It selects rank and ridge lambda exclusively from a held-out subset of **training features**; CIFAR-100 test features are used only after the selected configuration is locked.


In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'audit/t-soho-plan'  # change to your pushed branch/commit if needed
CHECKPOINT_SOURCE = 'huggingface'  # 'huggingface' (recommended) or 'google_drive'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
CACHE_DIR = '/content/tsoho_cifar100_cache'
OUTPUT_DIR = '/content/tsoho_cifar100_outputs'
SEED = 1993
NUM_TASKS = 10
BATCH_SIZE = 128
VALIDATION_FRACTION = 0.10
RANKS = '32,64'
RIDGE_LAMBDAS = '0.1,1.0'
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
import os, shutil, torch
!nvidia-smi
!rm -rf {WORK_DIR}
!git clone --branch {REPO_BRANCH} {REPO_GIT_URL} {WORK_DIR}
%cd {WORK_DIR}
!pip -q install -r requirements-kaggle.txt kagglehub huggingface_hub
if CHECKPOINT_SOURCE == 'google_drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
elif CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
else:
    raise ValueError("CHECKPOINT_SOURCE must be 'huggingface' or 'google_drive'")
print('checkpoint:', CHECKPOINT_PATH)
!git log -1 --oneline


In [ ]:
# Download only public CIFAR-100. Do NOT download CUB/ImageNet-R for this notebook.
from pathlib import Path
import kagglehub
downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
candidates = [downloaded, *downloaded.rglob('cifar-100')]
cifar_dir = next(p for p in candidates if (p / 'train').is_file() and (p / 'test').is_file() and (p / 'meta').is_file())
CIFAR_ROOT = str(cifar_dir)  # direct meta/train/test layout
print('CIFAR-100:', CIFAR_ROOT)
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH


In [ ]:
# Checkpoint identity, source tests, and one real preflight batch.
!python tools/checkpoint_preflight.py --root {CIFAR_ROOT} --checkpoint {CHECKPOINT_PATH} --checkpoint-size {CHECKPOINT_SIZE} --checkpoint-sha256 {CHECKPOINT_SHA256} --seed {SEED} --batch-size {BATCH_SIZE}
!python -m pytest -q tests/test_tsoho_learner.py tests/test_experiment_runner.py tests/test_tsoho_math.py tests/test_backbone_checkpoint.py


In [ ]:
# Extract frozen features once. Cache is infrastructure on disk, never learner state.
import os
if not os.path.exists(f'{CACHE_DIR}/metadata.json'):
    cmd = f'''python tools/experiment_runner.py --extract-features-only --root {CIFAR_ROOT} --backbone-checkpoint {CHECKPOINT_PATH} --backbone-checkpoint-size {CHECKPOINT_SIZE} --backbone-checkpoint-sha256 {CHECKPOINT_SHA256} --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/cache_extract --dataset CIFAR-100 --model-name vit_base_patch16_224 --data-augmentation vit --seed {SEED} --num-classes 100 --num-tasks {NUM_TASKS} --device cuda --batch-size {BATCH_SIZE} --num-workers 2'''
    assert os.system(cmd) == 0, cmd
else:
    print('Using validated existing cache:', CACHE_DIR)


In [ ]:
# Hyperparameter selection: only cached TRAIN features. It never opens test.pt.
selection_path = f'{OUTPUT_DIR}/selection.json'
cmd = f'''python tools/experiment_runner.py --select-config --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/selection --selection-output {selection_path} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --search-methods spectral_confusion_code --search-ranks {RANKS} --search-lambdas {RIDGE_LAMBDAS} --validation-fraction {VALIDATION_FRACTION}'''
assert os.system(cmd) == 0, cmd
import json
selected = json.load(open(selection_path))['best']
print('LOCKED train-only selection:', selected)


In [ ]:
# Final paired test evaluation. λ/rank are locked above; do not tune from these outputs.
methods = ['raw_ridge', 'random_orthogonal_code', 'truncated_simplex_code', 'spectral_confusion_code']
for method in methods:
    rank_arg = '' if method == 'raw_ridge' else f" --rank {selected['rank']}"
    run_dir = f"{OUTPUT_DIR}/final_{method}"
    cmd = f"python tools/experiment_runner.py --method {method}{rank_arg} --ridge-lambda {selected['ridge_lambda']} --feature-cache-dir {CACHE_DIR} --output-dir {run_dir} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --resume"
    assert os.system(cmd) == 0, cmd


In [ ]:
# Aggregate final metrics and download artifacts.
import glob, pandas as pd
rows=[]
for path in glob.glob(f'{OUTPUT_DIR}/final_*/metrics.json'):
    result=json.load(open(path)); result['method']=Path(path).parent.name.replace('final_',''); rows.append(result)
table=pd.DataFrame(rows).sort_values('method'); display(table)
!zip -r /content/tsoho_cifar100_artifacts.zip {OUTPUT_DIR}
from google.colab import files
files.download('/content/tsoho_cifar100_artifacts.zip')
